In [2]:
from pathlib import Path

import pandas as pd

In [3]:
node_table = pd.read_csv(snakemake.input.node_table)

In [4]:
node_table = pd.read_csv(snakemake.input.node_table)
public_metadata = pd.read_csv(snakemake.input.public_metadata)
rivm_metadata = pd.read_csv(snakemake.input.rivm_metadata)

In [5]:
node_table = node_table.rename(columns={
    'group_mash' : 'mash_cluster',
    'group_pling' : 'pling_subcommunity'
})

In [6]:
metadata = pd.concat([public_metadata, rivm_metadata], ignore_index=True)
metadata = node_table.merge(metadata, on="accession", how="left")
metadata.to_csv(snakemake.output.metadata, index=False)

In [7]:
print(metadata)

      accession pling_subcommunity mash_cluster nearest_neighbour  \
0    CP048644.1                  1            1        MT096435.1   
1    CP060142.1                  3            3        CP150655.1   
2    CP065195.1                  9            7        CP170580.1   
3    CP065516.1                  3            3        CP150655.1   
4    CP065798.1          Singleton            3        CP049977.1   
..          ...                ...          ...               ...   
227  LR134461.1          Singleton    Singleton        CP025986.1   
228  LT559118.1          Singleton    Singleton        AP026896.1   
229  OY725736.1          Singleton    Singleton        CP047217.1   
230  OY725841.1          Singleton    Singleton   pRIVM_M087195_3   
231  LR134422.1          Singleton    Singleton   pRIVM_M087195_3   

     mash_distance                      organism  \
0         0.000136         Staphylococcus aureus   
1         0.002165         Staphylococcus aureus   
2         0.012

In [8]:
def parse_finder(file):
    df = pd.read_table(file, sep="\t")

    if df.empty:
        df.loc[0] = None
    
    df["accession"] = Path(file).stem
    return df

In [9]:
resfinder = pd.concat([parse_finder(file) for file in snakemake.input.resfinder])

In [10]:
cfr_genes = (
    resfinder
    .fillna("")
    .groupby('accession')
    ['PRODUCT']
    .apply(lambda x: [y for y in x if 'cfr' in y.lower()])
    .apply(lambda x: ", ".join(sorted(set(x))))
    .replace("", None)
    .reset_index(name='cfr_gene')
)

In [11]:
def parse_tblastn(file):
    df = pd.read_csv(file)
    
    if df.empty:
        df.loc[0] = None
    
    df["accession"] = Path(file).stem
    df = (
        df
        .sort_values("evalue")
        .head(1)
    )
    df['cfr_like'] = df['qseqid'].apply(lambda x: x.split("|")[-1] + "-like" if pd.notnull(x) else None)
    return df[["accession", "cfr_like"]]

In [12]:
tblastn = pd.concat([parse_tblastn(file) for file in snakemake.input.tblastn], ignore_index=True)

In [13]:
tblastn.values

array([['pRIVM_M087195_3', None],
       ['pRIVM_M037203_1', 'cfrA-like'],
       ['pRIVM_M084526_3', 'cfrA-like'],
       ['pRIVM_M043455_2', 'cfrA-like'],
       ['pRIVM_M045402_1', 'cfrA-like'],
       ['pRIVM_M047916_1', 'cfrA-like'],
       ['pRIVM_M085090_1', 'cfrA-like'],
       ['pRIVM_M096426_1', 'cfrA-like'],
       ['pRIVM_M039053_1', 'cfrA-like'],
       ['pRIVM_M044329_1', 'cfrA-like'],
       ['pRIVM_C081951_1', 'cfrA-like'],
       ['pRIVM_M041248_1', 'cfrA-like'],
       ['pRIVM_M083782_1', 'cfrA-like'],
       ['pRIVM_M084986_4', 'cfrA-like'],
       ['AP026896.1', 'clbB-like'],
       ['CP002208.2', 'cfrA-like'],
       ['CP012031.1', 'clbB-like'],
       ['CP017783.1', 'clbC-like'],
       ['CP025986.1', 'cfrE-like'],
       ['CP028164.1', 'cfrA-like'],
       ['CP028728.1', 'cfrA-like'],
       ['CP028840.1', 'cfrA-like'],
       ['CP031260.1', 'cfrA-like'],
       ['CP032091.1', 'clbB-like'],
       ['CP034667.1', 'cfrA-like'],
       ['CP037908.1', None],
       [

In [14]:
cfr_genes = cfr_genes.merge(tblastn, on="accession")
cfr_genes["cfr_gene"] = cfr_genes["cfr_gene"].fillna(cfr_genes["cfr_like"])
cfr_genes = cfr_genes.fillna("None").drop(columns="cfr_like")

In [15]:
cfr_genes["data_source"] = cfr_genes["accession"].apply(lambda x: "RIVM" if "RIVM" in x else "Public")

In [16]:
(
    cfr_genes
    .fillna("None")
    .groupby("data_source")
    ["cfr_gene"]
    .value_counts()
    .unstack(level=0)
    .fillna("")
).to_csv(snakemake.output.cfr_count)

In [17]:
resistance_profile = (
    resfinder
    .fillna("")
    .groupby("accession")
    ["PRODUCT"]
    .apply(lambda x: ", ".join(sorted(set(x))))
    .reset_index(name="resistance profile")
)

In [18]:
resistance_profile = cfr_genes.merge(resistance_profile, on="accession")
resistance_profile.to_csv(snakemake.output.resistance, index=False)

In [19]:
def percent_count(group, column):
    unique_samples = group["accession"].nunique()
    counts = group[column].value_counts()
    counts = counts.apply(lambda x: f'{x / unique_samples:.0%} ({x}/{unique_samples})')
    return '; '.join([f'{a} {b}' for a, b in zip(counts.index, counts.values)])

In [20]:
mash_resistance = (
    resfinder
    .drop_duplicates(subset=["accession", "PRODUCT"])
    .merge(node_table, on="accession")
    .fillna("none")
    .groupby("mash_cluster")
    .apply(percent_count, "PRODUCT")
    .reset_index(name="resistance genes")
    .set_index("mash_cluster")
    )

/tmp/ipykernel_739505/1566556790.py:7: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(percent_count, "PRODUCT")


In [21]:
pling_resistance = (
    resfinder
    .drop_duplicates(subset=["accession", "PRODUCT"])
    .merge(node_table, on="accession")
    .fillna("none")
    .groupby("pling_subcommunity")
    .apply(percent_count, "PRODUCT")
    .reset_index(name="resistance genes")
    .set_index("pling_subcommunity")
    )

/tmp/ipykernel_739505/3398843347.py:7: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(percent_count, "PRODUCT")


In [22]:
mobtyper = pd.concat([parse_finder(file) for file in snakemake.input.mobtyper])

In [ ]:
mobtyper[['accession', 'predicted_mobility','rep_type(s)', 'relaxase_type(s)', 'mpf_type', "predicted_host_range_overall_name"]].to_csv(snakemake.output.mobility, index=False)

In [24]:
def median_iqr(x):
    return f'{x.median():.2f} [{x.quantile(.25):.2f}-{x.quantile(.75):.2f}]'

In [25]:
def count_agg(x):
    total = x.shape[0]
    x = x.fillna("None")
    counts = x.value_counts()
    counts = counts.apply(lambda x: f'{x / total:.0%} ({x}/{total})')
    return '; '.join([f'{a} {b}' for a, b in zip(counts.index, counts.values)])

In [26]:
mash_mob = (
    mobtyper
    .merge(node_table, on="accession")
    .fillna("none")
    .groupby("mash_cluster")
    .agg({
        'size' : median_iqr,
        'gc' : median_iqr,
        'predicted_mobility' : count_agg,
        'rep_type(s)' : count_agg,
        'relaxase_type(s)' : count_agg,
        'mpf_type' : count_agg,
        "predicted_host_range_overall_name" : count_agg
    })
)

In [27]:
pling_mob = (
    mobtyper
    .merge(node_table, on="accession")
    .fillna("none")
    .groupby("mash_cluster")
    .agg({
        'size' : median_iqr,
        'gc' : median_iqr,
        'predicted_mobility' : count_agg,
        'rep_type(s)' : count_agg,
        'relaxase_type(s)' : count_agg,
        'mpf_type' : count_agg,
        "predicted_host_range_overall_name" : count_agg
    })
)

In [28]:
mash_cfr = (
    cfr_genes
    .merge(node_table, on="accession")
    .fillna("none")
    .groupby("mash_cluster")
    .agg({
       "cfr_gene": count_agg
    })
)

In [29]:
pling_cfr = (
    cfr_genes
    .merge(node_table, on="accession")
    .fillna("none")
    .groupby("pling_subcommunity")
    .agg({
       "cfr_gene": count_agg
    })
)

In [ ]:
mash_metadata = (
    metadata
    .fillna("none")
    .groupby("mash_cluster")
    .agg({
       "genus": count_agg,
       "organism" : count_agg,
       "country" : count_agg
    })
)

In [ ]:
pling_metadata = (
    metadata
    .fillna("none")
    .groupby("pling_subcommunity")
    .agg({
       "genus": count_agg,
       "organism" : count_agg,
       "country" : count_agg
    })
)

In [ ]:
(
    mash_metadata
    .merge(mash_cfr, left_index=True, right_index=True)
    .merge(mash_mob, left_index=True, right_index=True)
    .merge(mash_resistance, left_index=True, right_index=True)
    ).to_csv(snakemake.output.mash_clusters)

In [ ]:
(
    pling_metadata
    .merge(pling_cfr, left_index=True, right_index=True)
    .merge(pling_mob, left_index=True, right_index=True)
    .merge(pling_resistance, left_index=True, right_index=True)
    ).to_csv(snakemake.output.pling_subcommunities)